# **Traditional RAG Pipeline**

### **1: Data Ingestion**

#### Step1: Instal Packages and Import Required Libraries

In [1]:
# Import required libraries
import os # To check and create file path 
import numpy as np # Use for handle numpy arrays
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader # PDF document loader from Langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter # ***
from pathlib import Path # ***
from sentence_transformers import SentenceTransformer # Embedding model library
import chromadb # Vector Database
from chromadb.config import Settings # ***
from sklearn.metrics.pairwise import cosine_similarity # Search for similarity
import uuid # ***
from typing import List, Dict, Any, Tuple # ***

/workspaces/Retrieval-Augmented-Generation-RAG_Basic/Traditional_RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Step2: Read and Load files from directory
For this practice we are going to use pdf files and pdf loader from langchain. It can be changed by any other files type loaders.

In [ ]:
# Function to read and load pdf files
def process_pdf_files(directory): #  Use "driectory" variable as input to indicate files directory path
    """ Process all PDF files in a directory"""

    all_documents = [] # Create an empty list to put all docs in it
    pdf_dir = Path(directory) # Put path of directory in a variable

    # Find all pdf files recursivaly
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process") # Print the length of pdf_files variable to know how many files we found to process

    # File loading process
    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name}") # print a message to start of loading process

        # Use pdf loder to process files
        try:
            loader = PyPDFLoader(str(pdf_file)) # Use PyPDFLoader for lightweight applications, simple text extraction
            # loader - PyMuPDFLoader(str(pdf_file)) # Use PyMuPDFLoader for large documents, and advanced feature extraction (tables, images, annotations)
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source-file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents) # appends all documnets from for loop to the end of the list
            print(f"Loaded {len(documents)} pages.")

        except Exception as e:
            print(f"Error: {e}") # print error message
    
    print(f"\n Total documents loaded: {len(all_documents)}") # Print number of loaded documnets
    return all_documents

# Call fuction to process all pdf files in the directory
all_pdf_documents = process_pdf_files("../data/pdf_files")

Found 8 PDF files to process

 Processing: Integrating Fine-Grained Audio-Visual Evidence for Robust Multimodal Emotion Reasoning.pdf
Loaded 6 pages.

 Processing: AVERE - Improving Audiovisual Emotion Reasoning with Preference Optimization.pdf
Loaded 11 pages.

 Processing: Emotion-LLaMAv2 and MMEVerse - A New Framework and Benchmark for Multimodal Emotion Understanding.pdf
Loaded 15 pages.

 Processing: Multimodal Classification Algorithms for Emotional Stress Analysis with an ECG-Centered Framework.pdf
Loaded 31 pages.

 Processing: XEmoGPT - An Explainable Multimodal Emotion Recognition Framework.pdf
Loaded 11 pages.

 Processing: HybridSense-LLM - A Structured Multimodal Framework for LLM Based Wellness Prediction.pdf
Loaded 19 pages.

 Processing: Improving cognitive stress classification via multimodal EEG and ECG fusion.pdf
Loaded 20 pages.

 Processing: EmoWork-A Multimodal Dataset for Assessing Emotion, Stress, and Emotional Workload.pdf
Loaded 14 pages.

 Total documents loa

#### Step3: Split documnets texts and creat chunks (Chunking)
We are going to use all documnets we created in previuse step and split the to chunks

In [ ]:
# Function to split text and chunking
def split_documents_to_text(documents, chunk_size=1000, chunk_overlap=200): # A function with 3 variables
    """ The function uses split technique to dvide documents into smaller text fpr better RAG performance.

    It Uses 3 variable: 1) "documets" from outsde of the functtion as input data, 2) "chunck_size" and "chunk_overlap"
 to define the size of each chunk and how many characters it allowed to have in common."""
    
    # Configuration of text spiter using "RecursiveCharacterTextSplitter" library
    text_spliter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", "", " "]
    )